# Machine Learning Pipeline: Catboost
**Predicting Bloodstream Infections**

This notebook is part of the official codebase for the paper: *"[Machine learning pipeline for predicting positive blood cultures using Sysmex hematology data in the emergency department.]"*. 
It contains the complete end-to-end machine learning pipeline for training, tuning, and evaluating the **Catboost** model.

### Overview of this notebook:
1. **Data Loading & Preprocessing:** Formatting the numerical dataset for model ingestion.
2. **Feature Selection:** Identifying predictive clinical variables (options: RFE, Boruta, Lasso, Tree Importance, or None).
3. **Hyperparameter Tuning:** Bayesian optimization using Optuna.
4. **Model Training & Calibration:** Training with cross-validation and applying Sigmoid probability calibration.
5. **Evaluation:** Extensive metric calculation, bootstrapping, SHAP value analysis, and threshold optimization.

**Repository & Citation:** 
For the full codebase, installation instructions, and citation details, please visit the [GitHub Repository](https://github.com/UMCG-SEH/blood-culture-outcome-classification).

*Note: Run all cells sequentially. Interactive prompts will appear at the bottom to configure the pipeline settings (e.g., device selection, validation split, SMOTE).*

In [132]:
import os
import json
from copy import deepcopy
from typing import List, Dict
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import joblib
import shap
from boruta import BorutaPy
from catboost import CatBoostClassifier
import optuna

from sklearn.model_selection import (
    StratifiedShuffleSplit,
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.calibration import cross_val_predict
from sklearn.exceptions import FitFailedWarning
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    recall_score,
    roc_curve,
    auc,
    make_scorer,
    precision_recall_curve,
    average_precision_score,
    precision_score,
    balanced_accuracy_score,
    f1_score,
    fbeta_score,
    matthews_corrcoef,
    log_loss
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import PredefinedSplit, GridSearchCV
from sklearn.model_selection import cross_validate

from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN, SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
import boruta
import optuna
from joblib import Parallel, delayed

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FitFailedWarning)


os.environ["CATBOOST_INFO_DIR"] = "catboost_tmp"

In [133]:
def make_output_folder(base: str, name: str) -> str:
    """
    Create a folder at the specified location if it doesn't exist.

    Args:
        base (str): Base directory where the folder should be created.
        name (str): Name of the folder to create.

    Returns:
        str: Full path to the created (or existing) folder.
    """
    path = os.path.join(base, name)
    os.makedirs(path, exist_ok=True)
    return path

In [134]:
def save_model(path: str, model) -> None:
    """
    Save a model object to the specified folder using joblib.

    Args:
        path (str): Folder path where the model will be saved.
        model: The model object to save.

    Returns:
        None
    """
    joblib.dump(model, os.path.join(path, 'model.joblib'))

In [135]:
def plot_roc(y_true, y_proba, out_path: str):
    """
    Plot the ROC curve for a set of true labels and predicted probabilities,
    and save it as 'roc.png' in the specified folder.

    Args:
        y_true: true binary labels.
        y_proba: predicted probabilities for the positive class.
        out_path (str): Folder path where the ROC plot will be saved.

    Returns:
        None
    """
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC curve (AUC = {roc_auc:.4f})')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, 'roc.png'))
    plt.close()

In [136]:
def plot_pr_curve(y_true, y_proba, out_path: str):
    """
    Plot the Precision-Recall curve for a set of true labels and predicted probabilities,
    and save it as 'pr_curve.png' in the specified folder.

    Args:
        y_true: true binary labels.
        y_proba: predicted probabilities for the positive class.
        out_path (str): Folder path where the PR curve plot will be saved.

    Returns:
        None
    """
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    ap_score = average_precision_score(y_true, y_proba)

    plt.figure()
    plt.plot(recall, precision, label=f'AP = {ap_score:.4f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall curve')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, 'pr_curve.png'))
    plt.close()

In [137]:
def plot_confusion(y_true, y_pred, out_path: str):
    """
    Plot a confusion matrix for true vs predicted labels and save it as 'confusion.png'.

    Args:
        y_true: true binary labels.
        y_pred: predicted binary labels.
        out_path (str): Folder path where the confusion matrix plot will be saved.

    Returns:
        None
    """
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title('Confusion matrix')
    plt.colorbar()

    classes = ['neg','pos']
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'), horizontalalignment="center")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, 'confusion.png'))
    plt.close()

In [138]:
def plot_feature_importances(feature_names, importances, out_path, top_n=20):
    """
    Plot feature importances, save the top features as a bar chart, and export all
    feature importances to a CSV file.

    Args:
        feature_names: list of feature names.
        importances: feature importance values corresponding to the features.
        out_path: Folder path where the outputs will be saved.
        top_n: Number of top features to display in the plot.

    Returns:
        None
    """
    os.makedirs(out_path, exist_ok=True)

    idx = np.argsort(importances)[::-1]
    names_sorted = [feature_names[i] for i in idx]
    vals_sorted = importances[idx]

    fi_df = pd.DataFrame({
        "feature": names_sorted,
        "importance": vals_sorted
    })
    fi_df.to_csv(
        os.path.join(out_path, "feature_importances.csv"),
        index=False
    )

    names = names_sorted[:top_n]
    vals = vals_sorted[:top_n]

    plt.figure(figsize=(8, max(4, top_n * 0.25)))
    plt.barh(range(len(vals))[::-1], vals)
    plt.yticks(range(len(vals))[::-1], names)
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, "feature_importances.png"))
    plt.close()

In [139]:
def plot_shap(pipeline, X, feature_names, out_path, top_n=20):
    """
    Compute SHAP values for a trained model pipeline, plot the summary of top N features,
    and save both the plot and SHAP values as a CSV file.

    Args:
        pipeline: Trained sklearn Pipeline with a step named 'cat' for the model.
        X: Input data used for SHAP computation.
        feature_names: List of feature names corresponding to X columns.
        out_path (str): Folder path where the plot and CSV will be saved.
        top_n (int, optional): Number of top features to display. Defaults 20.

    Returns:
        None
    """
    X = np.array(X)
    if X.ndim == 1:
        X = X.reshape(-1, 1)

    xg_trained = pipeline.named_steps['cat']
    explainer = shap.TreeExplainer(xg_trained)
    shap_values = explainer.shap_values(X)

    if isinstance(shap_values, list):
        shap_values_pos = shap_values[1]
    elif hasattr(shap_values, 'ndim') and shap_values.ndim == 3:
        shap_values_pos = shap_values[:, :, 1]
    else:
        shap_values_pos = shap_values

    n_features = X.shape[1]
    if shap_values_pos.shape[1] != n_features:
        shap_values_pos = shap_values_pos[:, :n_features]

    mean_abs_shap = np.mean(np.abs(shap_values_pos), axis=0)
    top_idx = np.argsort(mean_abs_shap)[::-1][:top_n]
    X_top = X[:, top_idx]
    shap_values_top = shap_values_pos[:, top_idx]
    feature_names_top = [feature_names[i] for i in top_idx]

    plt.figure()
    shap.summary_plot(shap_values_top, X_top, feature_names=feature_names_top, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, 'shap_summary_top15.png'))
    plt.close()

    shap_df = pd.DataFrame(shap_values_top, columns=feature_names_top)
    shap_df.to_csv(os.path.join(out_path, 'shap_values_top15.csv'), index=False)

In [140]:
def evaluate_model(clf, X_test, y_test) -> Dict[str, float]:
    """
    Evaluate a classification model on test data and return key metrics.

    Args:
        clf: Trained classifier with predict and predict_proba methods.
        X_test: Test features.
        y_test: True labels for the test set.

    Returns:
        Dict[str, float]: Dictionary containing AUC, accuracy, sensitivity, specificity,
                          confusion matrix counts (TP, TN, FP, FN), and combined metric.
    """
    y_prob = clf.predict_proba(X_test)[:, 1]
    y_pred = clf.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    sens = recall_score(y_test, y_pred, pos_label=1)
    spec = recall_score(y_test, y_pred, pos_label=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    combined = (sens + spec) / 2

    return {
        'auc': float(auc_score),
        'accuracy': float(acc),
        'sensitivity': float(sens),
        'specificity': float(spec),
        'tp': int(tp),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'combined': float(combined)
    }

In [141]:
def select_features(method, X_train, y_train, X_test, feature_columns, out_path=None, top_k=20, scoring_metric='roc_auc'):
    """
    Select features from training and test sets using different methods.

    Args:
        method: Feature selection method. Options: 'RFE', 'CAT_FEATURE_IMPORTANCE', 'BORUTA',
                      'TREE_IMPORTANCE', 'LASSO', 'NONE'.
        X_train: Training features (array-like or DataFrame).
        y_train: Training labels.
        X_test: Test features (array-like or DataFrame).
        feature_columns: List of feature names corresponding to X_train/X_test columns.
        out_path: Folder to save selected feature names. Defaults to None.
        top_k: Number of top features to select for importance-based methods. Defaults to 20.
        scoring_metric: Scoring metric for RFE. Defaults to 'roc_auc'.

    Returns:
        tuple: (X_train_sel, X_test_sel, selected_feat_names)
            X_train_sel: Selected training features.
            X_test_sel: Selected test features.
            selected_feat_names: List of selected feature names.
    """
    X_train_sel = np.array(X_train)
    X_test_sel = np.array(X_test)
    selected_feat_names = feature_columns
    method = method.upper()
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    if method == 'RFE':
        print(f"RFE selection started with scoring metric = {scoring_metric} ...")
        cat_est = CatBoostClassifier(eval_metric='AUC', train_dir=None, iterations=500, random_state=42)

        try:
            rfecv = RFECV(cat_est, step=1, cv=cv, scoring=scoring_metric, n_jobs=-1, min_features_to_select=2)
            rfecv.fit(X_train_sel, y_train)
        except ValueError as e:
            print(f"Scoring metric '{scoring_metric}' doesn't work for RFE. Using 'roc_auc' instead.")
            rfecv = RFECV(cat_est, step=1, cv=cv, scoring='roc_auc', n_jobs=-1, min_features_to_select=2)
            rfecv.fit(X_train_sel, y_train)

        mask = rfecv.support_
        selected_feat_names = [feature_columns[i] for i, m in enumerate(mask) if m]
        X_train_sel, X_test_sel = X_train_sel[:, mask], X_test_sel[:, mask]

    elif method == 'CAT_FEATURE_IMPORTANCE':
        print("CatBoost Feature Importance selection started...")
        cat = CatBoostClassifier(eval_metric='AUC', iterations=500, random_state=42, verbose=0)
        cat.fit(X_train_sel, y_train)
        importances = cat.feature_importances_
        idx = np.argsort(importances)[::-1][:top_k]
        selected_feat_names = [feature_columns[i] for i in idx]
        X_train_sel, X_test_sel = X_train_sel[:, idx], X_test_sel[:, idx]

    elif method == 'TREE_IMPORTANCE':
        print("Tree Importance selection started...")
        cat = CatBoostClassifier(eval_metric='AUC', iterations=350, random_state=42)
        cat.fit(X_train_sel, y_train)
        importances = cat.feature_importances_
        idx = np.argsort(importances)[::-1][:top_k]
        selected_feat_names = [feature_columns[i] for i in idx]
        X_train_sel, X_test_sel = X_train_sel[:, idx], X_test_sel[:, idx]

    elif method == 'BORUTA':
        print("Boruta selection started...")
        rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
        boruta_selector = BorutaPy(rf, n_estimators='auto', random_state=42, verbose=2, max_iter=500)
        boruta_selector.fit(X_train_sel, y_train)

        mask = boruta_selector.support_
        selected_feat_names = [feature_columns[i] for i, m in enumerate(mask) if m]
        X_train_sel, X_test_sel = X_train_sel[:, mask], X_test_sel[:, mask]

    elif method == 'LASSO':
        print("LASSO selection started...")
        lr = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=5000)
        lr.fit(X_train_sel, y_train)
        mask = np.abs(lr.coef_)[0] > 1e-6
        selected_feat_names = [feature_columns[i] for i, m in enumerate(mask) if m]
        X_train_sel, X_test_sel = X_train_sel[:, mask], X_test_sel[:, mask]

    elif method == 'NONE':
        print("No feature selection applied (all features used).")

    else:
        raise ValueError(f"Unknown feature selection method: {method}")

    if out_path and method not in ['NONE']:
        feat_df = pd.DataFrame({'selected_features': selected_feat_names})
        os.makedirs(out_path, exist_ok=True)
        feat_df.to_csv(os.path.join(out_path, f'selected_features_{method}.csv'), index=False)
        print(f"Selected features saved in: {os.path.join(out_path, f'selected_features_{method}.csv')}")

    if X_train_sel.ndim == 1:
        X_train_sel = X_train_sel.reshape(-1, 1)
        X_test_sel = X_test_sel.reshape(-1, 1)

    return X_train_sel, X_test_sel, selected_feat_names

In [142]:
def build_pipeline(smote_type=None, scale_pos_weight=1, categorical_features=None, outh_path=None, device='CPU'):
    """
    Build a machine learning pipeline with optional SMOTE resampling and a Catboost classifier.

    Args:
        smote_type (optional): Type of SMOTE resampling to apply. Options:
            'SMOTE', 'BorderlineSMOTE', 'ADASYN', 'SMOTENC'. Defaults to None.
        scale_pos_weight (optional): Class weight for the positive class in Catboost. Defaults to 1.
        categorical_features: Indices of categorical features for SMOTENC. Required if smote_type='SMOTENC'.
        device (optional): Device for Catboost training. Options: 'CPU', 'GPU'. Defaults to 'CPU'.

    Returns:
        ImbPipeline: Pipeline object including optional SMOTE, and Catboost classifier.
    """
    steps = []

    if smote_type is not None:
        if smote_type == 'SMOTE':
            sampler = SMOTE(random_state=42)
        elif smote_type == 'BorderlineSMOTE':
            sampler = BorderlineSMOTE(random_state=42)
        elif smote_type == 'ADASYN':
            sampler = ADASYN(random_state=42)
        elif smote_type == 'SMOTENC':
            if categorical_features is None:
                raise ValueError("For SMOTENC categorical_features indices must be specified.")
            sampler = SMOTENC(categorical_features=categorical_features, random_state=42)
        else:
            raise ValueError(f"unknown SMOTE type: {smote_type}")
        steps.append(('smote', sampler))

    cat = CatBoostClassifier(
        scale_pos_weight= scale_pos_weight,
        random_seed=42,
        thread_count=-1,
        loss_function='Logloss',
        boosting_type='Ordered',
        bootstrap_type='Bernoulli',
        verbose=False,
        task_type=device,
        devices='0',
        train_dir=os.path.join(outh_path, 'catboost_info') if outh_path else None
    )
    steps.append(('cat', cat))

    return ImbPipeline(steps=steps)

In [143]:
def get_scorer(scoring_metric):
    """
    Return an appropriate scorer object or string for use in model evaluation.

    Args:
        scoring_metric: Name of the scoring metric. Options include:
            'roc_auc', 'accuracy', 'f1', 'balanced_accuracy', 'f2',
            'mcc', 'recall', 'average_precision'/'ap', 'logloss'.

    Returns:
        str or callable: Scorer compatible with sklearn model selection methods.

    Raises:
        ValueError: If an unsupported scoring metric is provided.
    """
    scoring_metric = scoring_metric.lower()

    if scoring_metric == 'roc_auc':
        return 'roc_auc'
    elif scoring_metric == 'accuracy':
        return make_scorer(accuracy_score)
    elif scoring_metric == 'f1':
        return make_scorer(f1_score, average='binary')
    elif scoring_metric == 'balanced_accuracy':
        return make_scorer(balanced_accuracy_score)
    elif scoring_metric == 'f2':
        return make_scorer(fbeta_score, beta=2, average='binary')
    elif scoring_metric == 'mcc':
        return make_scorer(matthews_corrcoef)
    elif scoring_metric == 'recall':
        return make_scorer(recall_score, average='binary')
    elif scoring_metric in ['average_precision', 'ap']:
        return 'average_precision'
    elif scoring_metric == 'logloss':
        return 'neg_log_loss'
    else:
        raise ValueError(f"Unsupported scoring metric: {scoring_metric}")


In [144]:
def compute_score(y_true, y_pred, scoring_metric):
    """
    Compute a classification performance score based on the selected metric.

    Args:
        y_true: Array-like of true binary labels.
        y_pred: Array-like of predicted labels or probabilities, depending on the metric.
        scoring_metric: Metric to compute. Options include:
            'roc_auc', 'average_precision'/'ap', 'logloss', 'accuracy', 'f1',
            'balanced_accuracy', 'f2', 'mcc', 'recall'.

    Returns:
        Computed score for the given metric.

    Raises:
        ValueError: If an unsupported scoring metric is provided.
    """
    scoring_metric = scoring_metric.lower()

    if y_pred.ndim == 1 and np.any((y_pred > 0) & (y_pred < 1)):
        y_pred = (y_pred >= 0.5).astype(int)

    if scoring_metric == 'roc_auc':
        return roc_auc_score(y_true, y_pred)

    elif scoring_metric in ['average_precision', 'ap']:
        return average_precision_score(y_true, y_pred)

    elif scoring_metric == 'logloss':
        return log_loss(y_true, y_pred)

    elif scoring_metric == 'accuracy':
        return accuracy_score(y_true, y_pred)

    elif scoring_metric == 'f1':
        return f1_score(y_true, y_pred)

    elif scoring_metric == 'balanced_accuracy':
        return balanced_accuracy_score(y_true, y_pred)

    elif scoring_metric == 'f2':
        return fbeta_score(y_true, y_pred, beta=2)

    elif scoring_metric == 'mcc':
        return matthews_corrcoef(y_true, y_pred)

    elif scoring_metric == 'recall':
        return recall_score(y_true, y_pred)

    else:
        raise ValueError(f"Unsupported scoring_metric: {scoring_metric}")


In [145]:
def run_grid_search(
    pipeline, X_train, y_train,
    out_path=None, X_val=None, y_val=None,
    scoring_metric='roc_auc', n_trials=100, enable_pruning=False
):
    """
    Perform Optuna-based hyperparameter search for CatBoost classifier.

    Args:
        pipeline: sklearn-compatible pipeline with CatBoost classifier.
        X_train, y_train: Training data.
        out_path: Path to save best parameters CSV.
        X_val, y_val: Optional validation set.
        scoring_metric: Metric for evaluation.
        n_trials: Number of Optuna trials.
        enable_pruning: Enable Optuna pruning if True.

    Returns:
        Pipeline: Pipeline object fitted with best-found hyperparameters.
    """
    scorer = get_scorer(scoring_metric)

    if X_val is not None and y_val is not None:
        X_combined = np.vstack([X_train, X_val])
        y_combined = np.hstack([y_train, y_val])
        test_fold = -1 * np.ones(len(X_combined))
        test_fold[len(X_train):] = 0
        cv = PredefinedSplit(test_fold)
    else:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print("=== Start Optuna Search ===")

    pruner = optuna.pruners.MedianPruner() if enable_pruning else optuna.pruners.NopPruner()
    sampler = optuna.samplers.TPESampler(seed=42)

    def objective(trial):
        params = {
            'cat__iterations': trial.suggest_int('iterations', 300, 1500),
            'cat__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.5, log=True),
            'cat__depth': trial.suggest_int('depth', 3, 12),
            'cat__subsample': trial.suggest_float('subsample', 0.3, 1.0),
            'cat__l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 50),
            'cat__scale_pos_weight': trial.suggest_int('scale_pos_weight', 1, 20)
        }

        temp_pipe = clone(pipeline)
        temp_pipe.set_params(**params)

        if X_val is not None and y_val is not None:
            temp_pipe.fit(X_train, y_train)
            if hasattr(temp_pipe, "predict_proba") and scoring_metric.lower() in ['roc_auc', 'average_precision', 'ap']:
                y_val_pred = temp_pipe.predict_proba(X_val)[:, 1]
            else:
                y_val_pred = temp_pipe.predict(X_val)
            score = compute_score(y_val, y_val_pred, scoring_metric)
        else:
            scores = cross_val_score(temp_pipe, X_train, y_train, cv=cv, scoring=scorer, n_jobs=-1)
            score = scores.mean()

        trial.report(score, step=0)
        if enable_pruning and trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        return score

    study = optuna.create_study(direction="maximize", pruner=pruner, sampler=sampler)
    study.optimize(objective, n_trials=n_trials)

    print("\n=== OPTUNA RESULTS ===")
    print("Best Params:", study.best_params)
    print(f"Best CV {scoring_metric.upper()}: {study.best_value:.4f}")

    best_pipeline = clone(pipeline)
    fixed_params = {}

    for k, v in study.best_params.items():
        fixed_params[f"cat__{k}"] = v

    best_pipeline.set_params(**fixed_params)
    best_pipeline.fit(X_train, y_train)

    if out_path:
        df_optuna = pd.DataFrame([study.best_params])
        df_optuna[f'best_{scoring_metric}'] = study.best_value
        df_optuna.to_csv(os.path.join(out_path, 'optuna_best_params.csv'), index=False)

    if X_val is not None and y_val is not None:
        y_val_pred = best_pipeline.predict_proba(X_val)[:, 1] if hasattr(best_pipeline, "predict_proba") else best_pipeline.predict(X_val)
        val_score = compute_score(y_val, y_val_pred, scoring_metric)
        print(f"OPTUNA Validation {scoring_metric.upper()}: {val_score:.4f}")

    return best_pipeline

In [146]:
def evaluate_and_plot(best_pipeline, X_train_sel, X_test_sel, y_test, selected_feat_names, out_path, plots):
    """
    Evaluate a trained pipeline, save the model and metrics, and generate selected plots.

    Args:
        best_pipeline: Trained sklearn/ImbPipeline with Catboost classifier.
        X_train_sel: Training features used for SHAP plots.
        X_test_sel: Test features for evaluation.
        y_test: True labels for the test set.
        selected_feat_names: List of feature names corresponding to X_train_sel/X_test_sel.
        out_path (str): Output directory to save model, metrics, and plots.
        plots (list): List of plot types to generate. Options include:
            'roc', 'confusion', 'importance', 'shap', 'pr'.

    Returns:
        Evaluation metrics including AUC, accuracy, sensitivity, specificity, etc.
    """
    metrics = evaluate_model(best_pipeline, X_test_sel, y_test)
    save_model(out_path, best_pipeline)
    with open(os.path.join(out_path, 'metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)

    if 'roc' in plots:
        plot_roc(y_test, best_pipeline.predict_proba(X_test_sel)[:,1], out_path)
    if 'confusion' in plots:
        plot_confusion(y_test, best_pipeline.predict(X_test_sel), out_path)
    if 'importance' in plots:
        plot_feature_importances(selected_feat_names, best_pipeline.named_steps['cat'].feature_importances_, out_path)
    if 'shap' in plots:
        plot_shap(best_pipeline, X_train_sel, selected_feat_names, out_path)
    if 'pr' in plots:
        plot_pr_curve(y_test, best_pipeline.predict_proba(X_test_sel)[:,1], out_path)

    return metrics

In [147]:
def calculate_cv_metrics(model, X, y, cv=5):
    """
    Calculate multiple classification performance metrics using stratified
    cross-validation and return the mean and standard deviation for each metric.

    The function evaluates the model on each fold and computes discrimination,
    calibration-related, and diagnostic metrics based on predicted labels and
    probabilities.

    Args:
        model: Scikit-learn compatible classifier implementing fit, predict,
            and predict_proba.
        X: Feature matrix.
        y: Target labels.
        cv: Number of stratified cross-validation folds.

    Returns:
        results: Dictionary where each key is a metric name and each value
        is a dictionary containing the mean and standard deviation across folds.
    """
    if hasattr(X, "values"):
        X = X.values
    if hasattr(y, "values"):
        y = y.values

    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    metrics = {
        "roc_auc": [],
        "accuracy": [],
        "sensitivity_recall": [],
        "specificity": [],
        "precision": [],
        "balanced_accuracy": [],
        "f2_score": [],
        "youden_j_statistic": [],
        "lr_positive": [],
        "lr_negative": [],
        "diagnostic_odds_ratio": []
    }

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        sensitivity = recall_score(y_test, y_pred, pos_label=1)
        specificity = recall_score(y_test, y_pred, pos_label=0)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        j_stat = sensitivity + specificity - 1
        lr_pos = sensitivity / (1 - specificity) if specificity < 1 else np.inf
        lr_neg = (1 - sensitivity) / specificity if specificity > 0 else np.inf
        dor = lr_pos / lr_neg if lr_neg != 0 else np.inf

        metrics["roc_auc"].append(roc_auc_score(y_test, y_prob))
        metrics["accuracy"].append(accuracy_score(y_test, y_pred))
        metrics["precision"].append(precision_score(y_test, y_pred))
        metrics["sensitivity_recall"].append(sensitivity)
        metrics["specificity"].append(specificity)
        metrics["balanced_accuracy"].append(balanced_accuracy_score(y_test, y_pred))
        metrics["f2_score"].append(fbeta_score(y_test, y_pred, beta=2))
        metrics["youden_j_statistic"].append(j_stat)
        metrics["lr_positive"].append(lr_pos)
        metrics["lr_negative"].append(lr_neg)
        metrics["diagnostic_odds_ratio"].append(dor)

    results = {
        k: {
            "mean": np.mean(v),
            "std": np.std(v)
        } for k, v in metrics.items()
    }

    return results

In [148]:
def save_predictions_with_confidence(model, X_test_sel, y_test, out_path, threshold=0.5):
    """
    Generate predictions with confidence scores from a CatBoost model and save to CSV.

    Args:
        model: Trained CatBoost classifier with predict_proba method.
        X_test_sel: Test features for prediction.
        y_test: True labels corresponding to X_test_sel.
        out_path: Directory to save the CSV file.
        threshold: Probability cutoff for binary classification. Defaults to 0.5.

    Saves:
        CSV file 'predictions_with_confidence.csv' containing columns:
            - prediction: predicted class labels based on threshold
            - confidence: predicted probability for positive class
            - true_label: actual labels
    """
    y_proba = model.predict_proba(X_test_sel)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    df_out = pd.DataFrame({
        'prediction': y_pred,
        'confidence': y_proba,
        'true_label': y_test
    })
    df_out.to_csv(os.path.join(out_path, 'predictions_with_confidence.csv'), index=False)


In [149]:
def calculate_extended_metrics(model, X, y):
    """
    Compute a comprehensive set of classification metrics for a given model.

    Args:
        model: Trained sklearn-compatible classifier with predict and predict_proba methods.
        X: Features for evaluation.
        y: True binary labels.

    Returns:
        dict: Dictionary containing various performance metrics, including:
            - auc: ROC AUC score
            - accuracy
            - sensitivity (recall for positive class)
            - specificity (recall for negative class)
            - tp, tn, fp, fn: confusion matrix counts
            - combined: average of sensitivity and specificity
            - prc_auc: Area under precision-recall curve
            - f2: F2-score
            - lr+, lr-: positive and negative likelihood ratios
            - j_stat: Youden's J statistic
            - diagnostic_odds_ratio: ratio of lr+ to lr-
    """
    metrics = {}

    y_prob = model.predict_proba(X)[:, 1]
    metrics['auc'] = roc_auc_score(y, y_prob)

    y_pred = model.predict(X)
    metrics['accuracy'] = np.mean(y_pred == y)
    metrics['sensitivity'] = recall_score(y, y_pred, pos_label=1)
    metrics['specificity'] = recall_score(y, y_pred, pos_label=0)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    metrics['tp'] = tp
    metrics['tn'] = tn
    metrics['fp'] = fp
    metrics['fn'] = fn

    metrics['combined'] = (metrics['sensitivity'] + metrics['specificity']) / 2

    precision, recall, _ = precision_recall_curve(y, y_prob)
    metrics['prc_auc'] = auc(recall, precision)

    metrics['f2'] = fbeta_score(y, y_pred, beta=2)

    sens = metrics['sensitivity']
    spec = metrics['specificity']
    metrics['lr+'] = sens / (1 - spec) if spec < 1 else np.inf
    metrics['lr-'] = (1 - sens) / spec if spec > 0 else np.inf

    metrics['j_stat'] = sens + spec - 1

    metrics['diagnostic_odds_ratio'] = (
        metrics['lr+'] / metrics['lr-'] if metrics['lr-'] != 0 else np.inf
    )

    return metrics

In [150]:
def generate_threshold_confusion_matrices(out_path, custom_thresholds=None):
    """
    Generate confusion matrices and compute metrics for multiple thresholds
    using predictions with confidence scores saved in CSV. Saves plots and
    summaries to disk.

    Args:
        out_path: Directory containing 'predictions_with_confidence.csv'
            and where results will be saved.
        custom_thresholds (list, optional): List of additional thresholds to
            evaluate. Defaults to [0.1,0.2,0.3,0.4,0.6,0.7,0.8].

    Returns:
        dict: Summary of key thresholds and their Youden's J statistic, including:
            - best_threshold: threshold maximizing Youden's J
            - best_j: value of Youden's J at best_threshold
            - guaranteed_pos_threshold: minimum score for all positive cases
            - guaranteed_neg_threshold: maximum score for all negative cases
            - custom_thresholds: list of custom thresholds evaluated

    Saves:
        - Confusion matrix plots for best, guaranteed, and custom thresholds
        - CSV summary with metrics for all thresholds ('threshold_metrics_summary.csv')
        - JSON summary ('threshold_summary.json')
    """
    if custom_thresholds is None:
        custom_thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

    csv_path = os.path.join(out_path, "predictions_with_confidence.csv")
    df = pd.read_csv(csv_path)

    y_true = df["true_label"].values
    y_score = df["confidence"].values

    thresholds = np.linspace(0, 1, 1000)
    best_j = -np.inf
    best_t = 0.0
    for t in thresholds:
        y_pred = (y_score >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        sens = tp / (tp + fn) if (tp + fn) else 0
        spec = tn / (tn + fp) if (tn + fp) else 0
        j = sens + spec - 1
        if j > best_j:
            best_j = j
            best_t = t

    pos_mask = y_true == 1
    if pos_mask.sum() > 0:
        guaranteed_pos_threshold = float(np.min(y_score[pos_mask]))
    else:
        guaranteed_pos_threshold = 1.0

    neg_mask = y_true == 0
    if neg_mask.sum() > 0:
        max_neg = float(np.max(y_score[neg_mask]))
        eps = np.finfo(float).eps * 10
        guaranteed_neg_threshold = min(max_neg + eps, 1.0)
    else:
        guaranteed_neg_threshold = 0.0

    threshold_dir = os.path.join(out_path, "thresholded")
    os.makedirs(threshold_dir, exist_ok=True)

    def plot_cm(y_pred, thr, name, filename):
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(4,3))
        plt.imshow(cm, cmap="Blues")
        plt.title(f"{name}\nThreshold={thr:.6f}")
        plt.colorbar()
        plt.xticks([0,1], ["neg","pos"])
        plt.yticks([0,1], ["neg","pos"])
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, int(cm[i,j]), ha="center", va="center")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(threshold_dir, filename), dpi=300)
        plt.close()

    def compute_metrics(y_true, y_pred, y_prob=None):

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        sens = tp / (tp + fn) if (tp + fn) else 0
        spec = tn / (tn + fp) if (tn + fp) else 0
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        youden_j = sens + spec - 1

        lr_pos = sens / (1 - spec) if spec < 1 else np.inf
        lr_neg = (1 - sens) / spec if spec > 0 else np.inf
        dor = lr_pos / lr_neg if lr_neg != 0 else np.inf

        roc_auc = roc_auc_score(y_true, y_prob) if y_prob is not None else None

        return {
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "accuracy": acc,
            "sensitivity_recall": sens,
            "specificity": spec,
            "precision": prec,
            "F1": f1,
            "f2_score": f2,
            "balanced_accuracy": bal_acc,
            "youden_j_statistic": youden_j,
            "lr_positive": lr_pos,
            "lr_negative": lr_neg,
            "diagnostic_odds_ratio": dor,
            "roc_auc": roc_auc
        }

    thresholds_info = {
        "best_threshold": best_t,
        "guaranteed_positives": guaranteed_pos_threshold,
        "guaranteed_negatives": guaranteed_neg_threshold
    }

    summary_rows = []
    for name, t in thresholds_info.items():
        y_pred = (y_score >= t).astype(int)
        plot_cm(y_pred, t, name.replace("_", " ").title(), f"cm_{name}.png")
        metrics = compute_metrics(y_true, y_pred, y_prob=y_score)
        metrics["threshold"] = t
        metrics["type"] = name
        summary_rows.append(metrics)

    for t in custom_thresholds:
        y_pred = (y_score >= t).astype(int)
        plot_cm(y_pred, t, f"Custom Threshold {t}", f"cm_custom_{t:.2f}.png")
        metrics = compute_metrics(y_true, y_pred, y_prob=y_score)
        metrics["threshold"] = t
        metrics["type"] = "custom"
        summary_rows.append(metrics)

    df_summary = pd.DataFrame(summary_rows)
    csv_summary_path = os.path.join(threshold_dir, "threshold_metrics_summary.csv")
    df_summary.to_csv(csv_summary_path, index=False)

    summary_json = {
        "best_threshold": float(best_t),
        "best_j": float(best_j),
        "guaranteed_pos_threshold": float(guaranteed_pos_threshold),
        "guaranteed_neg_threshold": float(guaranteed_neg_threshold),
        "custom_thresholds": custom_thresholds
    }
    with open(os.path.join(threshold_dir, "threshold_summary.json"), "w") as f:
        json.dump(summary_json, f, indent=2)

    return summary_json

In [151]:
def bootstrap_model_performance(model, X_test, y_test, threshold=0.5, n_bootstrap=2000, n_jobs=-1, random_state=42):
    """
    Estimate model performance metrics with bootstrapped confidence intervals using parallel processing.

    Args:
        model: Trained classifier with predict_proba and predict methods.
        X_test: Test feature set.
        y_test: True binary labels for X_test.
        threshold: Probability cutoff for binary predictions.
        n_bootstrap: Number of bootstrap resamples.
        n_jobs: Number of parallel jobs. -1 uses all cores.
        random_state: Random seed.

    Returns:
        Bootstrapped metrics (mean, std, 95% CI) like before.
    """
    rng = np.random.RandomState(random_state)
    y_proba = model.predict_proba(X_test)[:, 1]
    N = len(y_test)

    def single_bootstrap(seed):
        local_rng = np.random.RandomState(seed)
        idx = local_rng.randint(0, N, N)
        y_bs = y_test[idx]
        proba_bs = y_proba[idx]
        pred_bs = (proba_bs >= threshold).astype(int)

        TP = np.sum((pred_bs == 1) & (y_bs == 1))
        TN = np.sum((pred_bs == 0) & (y_bs == 0))
        FP = np.sum((pred_bs == 1) & (y_bs == 0))
        FN = np.sum((pred_bs == 0) & (y_bs == 1))

        sens = TP / (TP + FN) if (TP + FN) > 0 else np.nan
        spec = TN / (TN + FP) if (TN + FP) > 0 else np.nan
        lr_pos = sens / (1 - spec) if (1 - spec) > 0 else np.nan
        lr_neg = (1 - sens) / spec if spec > 0 else np.nan
        dor = lr_pos / lr_neg if (lr_pos > 0 and lr_neg > 0) else np.nan

        return {
            "roc_auc": roc_auc_score(y_bs, proba_bs),
            "accuracy": accuracy_score(y_bs, pred_bs),
            "sensitivity_recall": sens,
            "specificity": spec,
            "precision": precision_score(y_bs, pred_bs, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_bs, pred_bs),
            "f2_score": fbeta_score(y_bs, pred_bs, beta=2, zero_division=0),
            "youden_j_statistic": sens + spec - 1,
            "lr_positive": lr_pos,
            "lr_negative": lr_neg,
            "diagnostic_odds_ratio": dor
        }

    results = Parallel(n_jobs=n_jobs)(
        delayed(single_bootstrap)(rng.randint(0, 1_000_000)) for _ in range(n_bootstrap)
    )

    def stats(metric):
        arr = np.array([r[metric] for r in results], dtype=float)
        return {
            "mean": float(np.nanmean(arr)),
            "std": float(np.nanstd(arr)),
            "ci_95": (
                float(np.nanpercentile(arr, 2.5)),
                float(np.nanpercentile(arr, 97.5))
            )
        }

    metrics_list = [
        "roc_auc", "accuracy", "sensitivity_recall", "specificity", "precision",
        "balanced_accuracy", "f2_score", "youden_j_statistic",
        "lr_positive", "lr_negative", "diagnostic_odds_ratio"
    ]

    return {metric: stats(metric) for metric in metrics_list} | {"threshold": threshold, "n_bootstrap": n_bootstrap}


In [ ]:
def run_experiments(
    df, feature_columns, target_col,
    outdir='cat_sysmex_runs',
    split_mode='validation'
):
    """
    Run a series of machine learning experiments using catboost classifiers
    with various feature selection methods, SMOTE strategies, and class weights.
    Trains models, evaluates performance, computes bootstrap confidence intervals,
    and generates plots and CSV/JSON summaries.

    Args:
        df: Input dataframe containing features and target.
        feature_columns: List of column names to use as features.
        target_col: Name of the target column in df.
        outdir: Base output directory for storing results. Defaults to 'cat_sysmex_runs'.
        split_mode : 'validation' to split training/validation/test sets,
            otherwise splits only training/test. Defaults to 'validation'.

    Returns:
        None. Saves outputs to disk, including:
            - Selected features per experiment
            - Trained models (catboost pipelines)
            - CV metrics on training data
            - Bootstrap metrics on test data
            - Predictions with confidence scores
            - Confusion matrix plots for thresholds
            - Extended evaluation metrics (AUC, accuracy, sensitivity, specificity, etc.)
            - Summary CSV ('summary.csv') aggregating results from all experiments

    Notes:
        - Supports different feature selection methods via `select_features`.
        - Supports SMOTE variations for handling class imbalance.
        - Supports hyperparameter tuning via `run_grid_search`.
        - Automatically calculates metrics, plots, and saves predictions for each experiment.
        - Summary table ranks models by AUC and combined sensitivity/specificity.
    """
    os.makedirs(outdir, exist_ok=True)
    summary_rows = []

    X = df[feature_columns].values
    y = df[target_col].values

    if split_mode == 'validation':
        X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
        X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, stratify=y_temp, random_state=42)
    else:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
        X_val, y_val = None, None

    for method in feature_methods:
        for smote_type in smote_types:
            folder_name = f"cat_{method.lower()}_{smote_type.lower()}"
            out_path = os.path.join(outdir, folder_name)
            os.makedirs(out_path, exist_ok=True)

            if X_val is not None:
                X_train_sel, X_val_sel, selected_feat_names = select_features(
                    method, X_train, y_train, X_val, feature_columns, out_path=out_path, scoring_metric=scoring_metric
                )
            else:
                X_train_sel, _, selected_feat_names = select_features(
                    method, X_train, y_train, X_test, feature_columns, out_path=out_path, scoring_metric=scoring_metric
                )
                X_val_sel = None

            categorical_features = []
            pipeline = build_pipeline(smote_type if smote_type != 'None' else None, categorical_features)

            best_pipeline = run_grid_search(
                pipeline=pipeline,
                X_train=X_train_sel,
                y_train=y_train,
                out_path=out_path,
                X_val=X_val_sel,
                y_val=y_val,
                scoring_metric=scoring_metric,
                n_trials=n_trials,
                enable_pruning=enable_pruning
            )

            cv_metrics = calculate_cv_metrics(best_pipeline, X_train_sel, y_train)
            with open(os.path.join(out_path, "cv_metrics.json"), "w") as f:
                json.dump(cv_metrics, f, indent=2)

            if X_val_sel is not None:
                X_combined = np.vstack([X_train_sel, X_val_sel])
                y_combined = np.hstack([y_train, y_val])
            else:
                X_combined, y_combined = X_train_sel, y_train

            best_pipeline.fit(X_combined, y_combined)

            clf_step_name = 'cat'
            original_clf = best_pipeline.named_steps[clf_step_name]
            calibrated_clf = CalibratedClassifierCV(original_clf, method='sigmoid', cv='prefit')
            calibrated_clf.fit(X_combined, y_combined)
            best_pipeline.named_steps[clf_step_name] = calibrated_clf
            print(f"Model '{folder_name}' gecalibreerd met sigmoid method.")

            def get_feature_importances(pipeline, step_name):
                clf = pipeline.named_steps[step_name]
                if isinstance(clf, CalibratedClassifierCV):
                    clf = clf.base_estimator
                return clf.feature_importances_

            importances = get_feature_importances(best_pipeline, clf_step_name)

            X_test_sel = X_test[:, [feature_columns.index(f) for f in selected_feat_names]]

            bootstrap_results = bootstrap_model_performance(best_pipeline, X_test_sel, y_test, n_bootstrap=2000)
            with open(os.path.join(out_path, "bootstrap_metrics.json"), "w") as f:
                json.dump(bootstrap_results, f, indent=2)
            print("Bootstrap CI opgeslagen.")

            evaluate_and_plot(best_pipeline, X_combined, X_test_sel, y_test, selected_feat_names, out_path, plots)
            plot_feature_importances(selected_feat_names, importances, out_path)
            metrics = calculate_extended_metrics(best_pipeline, X_test_sel, y_test)
            save_predictions_with_confidence(best_pipeline, X_test_sel, y_test, out_path, threshold=0.5)
            generate_threshold_confusion_matrices(out_path)

            summary_rows.append({
                **metrics,
                'folder': folder_name,
                'method': method,
                'smote_type': smote_type,
                'n_features': len(selected_feat_names),
                'split_mode': split_mode
            })

    df_summary = pd.DataFrame(summary_rows).sort_values(by=['auc', 'combined'], ascending=False)
    df_summary.to_csv(os.path.join(outdir, 'summary.csv'), index=False)
    print(f"All models have been trained and results saved in '{outdir}'")

In [ ]:
def safe_input(prompt, valid_values=None, default=None, type_func=str, allow_multiple=False):
    """
    Prompt the user for input safely with optional validation, default values, and type conversion.

    Args:
        prompt: The message to display to the user.
        valid_values (optional): List of allowed values. If provided, input not in this list
            will fall back to the default.
        default: The default value to return if the user enters nothing or invalid input.
        type_func: Function to convert the input string to a desired type
            (e.g., int, float, str). Defaults to str.
        allow_multiple: If True, allows multiple values separated by spaces,
            returning a list of converted items. Defaults to False.

    Returns:
        single value or list: The user input converted to the desired type, or default if invalid/missing.

    Notes:
        - If allow_multiple=True, the input string is split on spaces, each item is type-converted,
          and checked against valid_values if provided.
        - If the user input is empty, the default is returned.
        - If conversion or validation fails, the default is returned and a message is printed.
    """
    val = input(f"{prompt} [default: {default}]: ").strip()
    if not val:
        return default
    try:
        if allow_multiple:
            items = [type_func(v) for v in val.split()]
            if valid_values:
                items = [v if v in valid_values else default for v in items]
            return items
        else:
            typed_val = type_func(val)
            if valid_values and typed_val not in valid_values:
                print(f"Invalid input '{typed_val}', using default '{default}'.")
                return default
            return typed_val
    except Exception as e:
        print(f"Invalid input '{val}' ({e}), using default '{default}'.")
        return default

device_choice = safe_input(
    "Which device do you want to use for catboost? (cpu / gpu)",
    valid_values=['cpu', 'gpu'],
    default='cpu',
    type_func=str
)

split_mode = safe_input(
    "Do you want to use a validation set? (none / validation)",
    valid_values=['none', 'validation'],
    default='validation'
)

fs_input = safe_input(
    "Which feature selection method? (NONE / RFE / BORUTA / TREE_IMPORTANCE / LASSO / all)",
    default='NONE RFE BORUTA',
    type_func=str,
    allow_multiple=True
)
feature_methods = [s.upper() for s in fs_input] if isinstance(fs_input, list) else fs_input.upper().split()
if 'ALL' in feature_methods:
    feature_methods = ['NONE','RFE','BORUTA','TREE_IMPORTANCE','LASSO']

plots_input = safe_input(
    "Which results/plots do you want? (roc, pr, confusion, importance, shap, all)",
    default='roc pr confusion importance shap',
    type_func=str,
    allow_multiple=True
)
plots = [p.lower() for p in plots_input] if isinstance(plots_input, list) else plots_input.lower().split()
if 'all' in plots:
    plots = ['roc','pr','confusion','importance','shap']


metric_input = safe_input(
    "Which scoring metric do you want to use? "
    "(roc_auc / average_precision / f2 / balanced_accuracy / mcc / f1 / accuracy / recall)",
    valid_values=['roc_auc', 'average_precision', 'f2', 'balanced_accuracy', 'mcc', 'f1', 'accuracy','recall'],
    default='roc_auc',
    type_func=str
)
scoring_metric = metric_input

grid_type_input = safe_input(
    "Which grid search do you want to use? (optuna)",
    valid_values=['optuna'],
    default='optuna',
    type_func=str
)
grid_type = 'grid' if grid_type_input == 'standard' else grid_type_input


if grid_type == 'optuna':
    n_trials = safe_input("Number of Optuna trials?", default=100, type_func=int)
    enable_pruning = safe_input("Enable Optuna pruning? (yes/no)", valid_values=['yes','no'], default='yes') == 'yes'

smote_types = safe_input(
    "Which SMOTE version do you want to use? (SMOTE / BorderlineSMOTE / ADASYN / SMOTENC / None)",
    default=['None'],
    type_func=str,
    allow_multiple=True
)

In [ ]:
df = pd.read_csv('final_training.csv')
target_col = 'status'
feature_columns = [c for c in df.columns if c != target_col]

In [ ]:
run_experiments(df, feature_columns, target_col, outdir='cat_sysmex_runs', split_mode=split_mode)
print("All models are trained and saved in 'cat_sysmex_runs'.")